# v9: clearly organized controllers — same v4 driving behavior

Based directly on `v4_repaired.ipynb`, not v7/v8. This is a readability refactor: motor settings, vision, timings, marker rules and recovery behavior are retained. Camera-only turn completion does not measure a 90-degree angle.

## Start here: one decision maker, three responsibilities

| Component | Plain-language responsibility | When it owns the wheel commands |
|---|---|---|
| **NavigationCoordinator** | Decide who is in control and latch failures | Every processed frame; does not write to motors |
| **GreenMarkerTurnController** | Confirm green instructions and perform the requested turn | During stop/advance/turn, including the completion iteration |
| **GapCrossingController** | Recognize a plausible white gap and cross it briefly | During a justified gap crossing |
| **BlackLineController** | Follow black tape, negotiate bends and search briefly for lost tape | All other ordinary driving |
| **Runner (`run_v9`)** | Check frame freshness, call the coordinator, update the display and apply commands | Sole motor-command writer during a run |

Decision priority: **continue an active green maneuver → accept an eligible confirmed green request → handle a gap → follow/search the black line**. Eligibility preserves v4: new green maneuvers cannot interrupt an active gap or sharp bend. The first two marker-confirmation frames still use normal line/gap control.

GapCrossingController uses composition: it owns a BlackLineController and delegates to it when not crossing a gap. It also observes ordinary following to gather straight-line evidence. This is why the code is not simply three independent controllers selected from one frame.

The diagram below is embedded SVG so it displays in ordinary Jupyter without a Mermaid extension. It is a high-level ownership diagram, not the detailed internal state machine of each controller. Run its display cell while stationary, then proceed through the notebook one cell at a time.


In [ ]:
from IPython.display import SVG, display

display(SVG('<svg xmlns="http://www.w3.org/2000/svg" width="1060" height="620" viewBox="0 0 1060 620">\n<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 Z" fill="#475569"/></marker></defs>\n<rect width="1060" height="620" fill="#f8fafc"/>\n<style>text{font-family:Arial,sans-serif;fill:#172033;font-size:16px}.title{font-size:23px;font-weight:bold}.head{font-weight:bold;font-size:18px}.small{font-size:14px}.box{stroke:#64748b;stroke-width:1.5;rx:12}.arrow{fill:none;stroke:#475569;stroke-width:2;marker-end:url(#arrow)}</style>\n<text x="35" y="38" class="title">V9: who controls the robot on each new frame?</text>\n<rect x="320" y="62" width="420" height="64" fill="#e2e8f0" class="box"/>\n<text x="530" y="88" text-anchor="middle" class="head">Runner: validate camera and run time</text>\n<text x="530" y="111" text-anchor="middle">Vision supplies line positions + marker direction</text>\n<path d="M530 126 V168" class="arrow"/>\n<rect x="320" y="170" width="420" height="74" fill="#dbeafe" class="box"/>\n<text x="530" y="198" text-anchor="middle" class="head">NavigationCoordinator</text>\n<text x="530" y="222" text-anchor="middle">Choose one owner using the priority below</text>\n<path d="M385 244 V275 H180 V316" class="arrow"/>\n<path d="M530 244 V316" class="arrow"/>\n<path d="M675 244 V275 H880 V316" class="arrow"/>\n<text x="180" y="302" text-anchor="middle" class="small">1. Active / confirmed eligible green instruction</text>\n<text x="530" y="302" text-anchor="middle" class="small">2. Otherwise: active / justified gap</text>\n<text x="880" y="302" text-anchor="middle" class="small">3. Otherwise: black-line driving</text>\n<rect x="25" y="318" width="310" height="98" fill="#dcfce7" class="box"/>\n<text x="180" y="347" text-anchor="middle" class="head">GreenMarkerTurnController</text>\n<text x="180" y="373" text-anchor="middle">Stop → advance → turn → return</text>\n<text x="180" y="397" text-anchor="middle" class="small">Owns marker confirmation and cooldown</text>\n<rect x="375" y="318" width="310" height="98" fill="#fef3c7" class="box"/>\n<text x="530" y="347" text-anchor="middle" class="head">GapCrossingController</text>\n<text x="530" y="373" text-anchor="middle">Cross briefly → confirm tape → return</text>\n<text x="530" y="397" text-anchor="middle" class="small">Delegates normal driving to black-line control</text>\n<rect x="725" y="318" width="310" height="98" fill="#e2e8f0" class="box"/>\n<text x="880" y="347" text-anchor="middle" class="head">BlackLineController</text>\n<text x="880" y="373" text-anchor="middle">Follow → sharp bend / search → follow</text>\n<text x="880" y="397" text-anchor="middle" class="small">Ordinary tape steering and bounded recovery</text>\n<path d="M180 416 V449 H530 V480" class="arrow"/>\n<path d="M530 416 V480" class="arrow"/>\n<path d="M880 416 V449 H530" fill="none" stroke="#475569" stroke-width="2"/>\n<rect x="320" y="482" width="420" height="62" fill="#dbeafe" class="box"/>\n<text x="530" y="508" text-anchor="middle" class="head">Return left / right commands + status</text>\n<text x="530" y="532" text-anchor="middle">Runner applies only in DRIVE; repeat next frame</text>\n<text x="35" y="580" class="head">Any failure or expired run → stop motors.</text>\n<text x="35" y="605" class="small">Controller failures latch until a new run. A new green turn cannot interrupt an active gap or sharp bend.</text>\n</svg>'))


## How to read the code

1. **Perception answers “what is visible?”** It returns three line offsets and a marker direction. It never moves the robot.
2. **NavigationCoordinator answers “who should act?”** It first offers the frame to the green controller. A return value of `None` means “I do not own this frame.” A command tuple means “use this maneuver's output.”
3. **GapCrossingController answers “is this a justified gap?”** It remembers recent straight-line evidence. Outside a crossing, it asks BlackLineController for ordinary driving commands.
4. **The selected controller returns `(left, right, detail_state)`.** It does not access motor hardware.
5. **The runner applies the result only in driving mode.** Preview computes the same result without applying drive commands. The status adds `owner=BLACK_LINE`, `GAP_CROSSING`, `GREEN_TURN`, or `STOPPED`.

After successful green completion, the coordinator creates fresh line/gap controllers to discard old turn history. The green controller itself stays alive to preserve cooldown. During `GREEN_DONE`, its zero command still owns that iteration; normal following resumes on the next frame.

### Names changed from v4

- `CurveController` → `BlackLineController`.
- `GapCurveController` → `GapCrossingController`; inheritance replaced by explicit delegation.
- `GreenNavigator` → split into `GreenMarkerTurnController` and `NavigationCoordinator`.
- `run_v4` → `run_v9`; `run_preview` and `run_robot` keep their familiar names.

**Behavior intentionally retained:** three image bands; stationary inner wheel on ordinary pivots; reversing inner wheel on green turns; 0.15-second green advance; BOTH-marker U-turn; gap deadlines and marker cooldown. Internal detail labels such as FOLLOW, SEARCH and GREEN_ADVANCE are retained for debugging. Detailed per-controller state diagrams can be added next.

No driving starts automatically. Run cells individually, stop previous camera kernels first, and copy your proven calibration before floor testing. Never run cleanup concurrently with a driving loop.


## 1. Hardware and camera
Initialize the robot stopped and display the camera.


In [ ]:
import time
import cv2
import numpy as np
import traitlets
import ipywidgets as widgets

from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

camera = Camera.instance(width=224, height=224)

live_view = widgets.Image(format="jpeg", width=336, height=336) # Live camera display

camera_link = traitlets.dlink(
    (camera, "value"),
    (live_view, "value"),
    transform=bgr8_to_jpeg
)

display(live_view)


In [ ]:
latest = {"sample": None} # Latest image and its arrival time

def receive_frame(change):
    latest["sample"] = (change["new"], time.monotonic()) # Keep image and timestamp together

camera.observe(receive_frame, names="value") # Observe new camera frames

time.sleep(0.5) # Allow the first callback to arrive

if latest["sample"] is None:
    print("No camera frame yet.")
else:
    print("Camera frame received.")


## 2. Perception: detect black tape and green markers
Unchanged v4 vision. These functions return observations, not motor commands.


In [ ]:
BLACK_THRESHOLD = 70  # Copy your calibrated threshold here.
# Fractions of image height: change these if the far strip is above the floor.
BANDS = [("near", 0.72, 0.88), ("middle", 0.54, 0.70), ("far", 0.36, 0.52)]

def detect_curve(frame):
    height, width = frame.shape[:2]
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
    overlay = frame.copy()
    positions = {}
    for name, top, bottom in BANDS:
        y0, y1 = int(top * height), int(bottom * height)
        strip = mask[y0:y1, :]
        contours = cv2.findContours(strip, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
        positions[name] = None
        if contours:
            contour = max(contours, key=cv2.contourArea)
            fraction = cv2.contourArea(contour) / float(strip.size)
            moments = cv2.moments(contour)
            # Permit a broad horizontal segment, but reject an almost dark strip.
            if 0.005 < fraction < 0.80 and moments["m00"] > 0:
                cx = moments["m10"] / moments["m00"]
                cy = int(moments["m01"] / moments["m00"]) + y0
                positions[name] = (cx - width / 2.0) / (width / 2.0)
                cv2.circle(overlay, (int(cx), cy), 4, (0, 255, 0), -1)
        cv2.rectangle(overlay, (0, y0), (width - 1, y1), (255, 0, 0), 1)
        cv2.putText(overlay, name, (3, y0 + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 255), 1)
    cv2.line(overlay, (width // 2, 0), (width // 2, height - 1), (0, 255, 255), 1)
    return positions, overlay, mask

# OpenCV HSV hue range is 0..179. Tune these using the actual green tape.
GREEN_LOWER = np.array([35, 70, 50], dtype=np.uint8)
GREEN_UPPER = np.array([90, 255, 255], dtype=np.uint8)
GREEN_MIN_AREA = 40
GREEN_MAX_AREA = 3000
GREEN_ROI_TOP = 0.45
GREEN_ROI_BOTTOM = 0.92
GREEN_TRIGGER_Y = 0.65  # Require a nearby marker before initiating a turn.
GREEN_SIDE_MARGIN = 8.0
GREEN_MAX_SIDE_DISTANCE = 65.0  # Pixels, at the configured 224px resolution.
GREEN_PAIR_Y_TOLERANCE = 0.15

def detect_green(frame):
    height, width = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, GREEN_LOWER, GREEN_UPPER)
    mask[:int(height * GREEN_ROI_TOP), :] = 0
    mask[int(height * GREEN_ROI_BOTTOM):, :] = 0
    kernel = np.ones((3, 3), dtype=np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    markers = []
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        if not GREEN_MIN_AREA <= area <= GREEN_MAX_AREA or h == 0:
            continue
        if not 0.4 <= w / float(h) <= 2.5:
            continue
        moments = cv2.moments(contour)
        if moments['m00'] <= 0:
            continue
        markers.append(dict(x=moments['m10']/moments['m00'],
                            y=moments['m01']/moments['m00'],
                            area=area, w=w, h=h, box=(x,y,w,h)))
    markers.sort(key=lambda item: item['area'], reverse=True)
    return markers, mask

def get_line_points(positions, width, height):
    band_y = {name: (top + bottom) / 2.0 for name, top, bottom in BANDS}
    return [(width * (1.0 + positions[name]) / 2.0, height * band_y[name])
            for name in ('far', 'middle', 'near') if positions.get(name) is not None]

def classify_green_direction(positions, markers, width, height):
    if not markers:
        return 'NONE'
    points = get_line_points(positions, width, height)
    if len(points) < 2:
        return 'UNKNOWN'
    near_x, near_y = points[-1]
    dx, dy = points[0][0] - near_x, points[0][1] - near_y
    length = (dx * dx + dy * dy) ** 0.5
    if length < 1e-6:
        return 'UNKNOWN'
    dx, dy = dx / length, dy / length
    sides = {'LEFT': [], 'RIGHT': []}
    for marker in markers:
        if marker['y'] < height * GREEN_TRIGGER_Y:
            continue
        score = (marker['x'] - near_x) * dy - (marker['y'] - near_y) * dx
        if GREEN_SIDE_MARGIN < abs(score) <= GREEN_MAX_SIDE_DISTANCE:
            sides['LEFT' if score > 0 else 'RIGHT'].append(marker)
    if sides['LEFT'] and sides['RIGHT']:
        paired = any(abs(a['y'] - b['y']) <= height * GREEN_PAIR_Y_TOLERANCE
                     for a in sides['LEFT'] for b in sides['RIGHT'])
        return 'BOTH' if paired else 'UNKNOWN'
    if sides['LEFT']:
        return 'LEFT'
    if sides['RIGHT']:
        return 'RIGHT'
    return 'CENTER'


## 3. BlackLineController: follow black tape
Owns proportional steering, sharp bends and lost-line search. Internal behavior is unchanged.


In [ ]:
# Copy your measured, reliable motor settings. These are initial guesses.
BASE_SPEED = 0.15
CURVE_SPEED = 0.13  # Must still reliably move the robot on the floor.
KP = 0.14
MAX_SPEED = 0.25
TURN_SPEED = 0.15
ALLOW_REVERSE = False  # True enables an in-place turn with one wheel reversing.
TURN_ENTER = 0.55
TURN_EXIT = 0.22
TURN_TIMEOUT = 1.0
LOST_TIMEOUT = 0.35
FRAME_TIMEOUT = 0.5

def clamp(value, low, high):
    return max(low, min(high, value))

class BlackLineController:
    """Follow black tape, handle sharp bends and search briefly when tape is lost."""
    def __init__(self):
        self.turn_direction = 0
        self.turn_started = None
        self.lost_started = None
        self.last_direction = 0
        self.stopped = False

    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def pivot(self, direction):
        speed = clamp(TURN_SPEED, 0.0, MAX_SPEED)
        inner = -speed if ALLOW_REVERSE else 0.0
        return (speed, inner) if direction > 0 else (inner, speed)

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        visible = [(name, value) for name, value in positions.items() if value is not None]

        if not visible:
            if self.lost_started is None:
                self.lost_started = now
            if now - self.lost_started >= LOST_TIMEOUT:
                return self.stop("line not reacquired")
            if not self.last_direction:
                return self.stop("line lost without a known turn direction")
            if self.turn_started is None:
                self.turn_started = now
                self.turn_direction = self.last_direction
            if now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")
            left, right = self.pivot(self.turn_direction)
            return left, right, "SEARCH"

        self.lost_started = None
        weights = {"near": 0.45, "middle": 0.35, "far": 0.20}
        error = sum(weights[name] * value for name, value in visible) / sum(weights[name] for name, _ in visible)
        farthest = positions.get("far")
        if farthest is None:
            farthest = positions.get("middle")
        # Anticipate a bend from its displacement relative to the near strip.
        if near is not None and farthest is not None:
            error += 0.40 * (farthest - near)
        error = clamp(error, -1.0, 1.0)
        if abs(error) > 0.15:
            self.last_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            # Require the near line and combined direction to agree before exiting.
            if near is not None and abs(near) < TURN_EXIT and abs(error) < TURN_EXIT:
                self.turn_started = None
                self.turn_direction = 0
            elif now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")

        if self.turn_started is None and abs(error) >= TURN_ENTER:
            self.turn_started = now
            self.turn_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        bend = min(abs(error) / TURN_ENTER, 1.0)
        base = BASE_SPEED + (CURVE_SPEED - BASE_SPEED) * bend
        correction = KP * error
        left = clamp(base + correction, 0.0, MAX_SPEED)
        right = clamp(base - correction, 0.0, MAX_SPEED)
        return left, right, "FOLLOW"


## 4. GapCrossingController: handle short white gaps
Owns straight-line evidence and gap crossing. Uses a BlackLineController for ordinary driving instead of inheriting from it.


In [ ]:
# Initial trial values; calibrate on the real floor at your working motor power.
GAP_SPEED = BASE_SPEED
GAP_TIMEOUT = 1.0       # Maximum time driving without confirmed near/middle tape.
GAP_STEER_LIMIT = 0.015 # Only a small remembered correction; not heading feedback.
GAP_APPROACH_FRAMES = 3
GAP_APPROACH_MAX_AGE = 0.6
GAP_ALIGNMENT = 0.25   # Normalized offset from image centre.
GAP_STRAIGHT_SPREAD = 0.12
GAP_CONFIRM_FRAMES = 2

class GapCrossingController:
    """Own gap evidence/crossing; delegate ordinary driving to BlackLineController."""
    def __init__(self):
        self.black_line = BlackLineController()
        self.straight_frames = 0
        self.straight_seen = None
        self.saved_correction = 0.0
        self.gap_started = None
        self.confirm_frames = 0

    def gap_commands(self):
        return (clamp(GAP_SPEED + self.saved_correction, 0.0, MAX_SPEED),
                clamp(GAP_SPEED - self.saved_correction, 0.0, MAX_SPEED),
                "GAP: forward, awaiting tape")

    def step(self, positions, now):
        if self.black_line.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        middle = positions.get("middle")
        values = [value for value in positions.values() if value is not None]

        if self.gap_started is not None:
            # Never reset this deadline because of an isolated noisy detection.
            if now - self.gap_started >= GAP_TIMEOUT:
                return self.black_line.stop("gap crossing timed out")
            if near is not None and middle is not None:
                self.confirm_frames += 1
            else:
                self.confirm_frames = 0
            if self.confirm_frames < GAP_CONFIRM_FRAMES:
                return self.gap_commands()
            self.gap_started = None
            self.confirm_frames = 0
            self.black_line.lost_started = None
            self.black_line.last_direction = 0
            # Resume the existing controller using the newly observed tape.

        if not values:
            recent_straight = (self.straight_seen is not None and
                               now - self.straight_seen <= GAP_APPROACH_MAX_AGE)
            if (self.black_line.turn_started is None and recent_straight and
                    self.straight_frames >= GAP_APPROACH_FRAMES):
                self.gap_started = now
                self.straight_frames = 0
                self.straight_seen = None
                self.confirm_frames = 0
                return self.gap_commands()
            return self.black_line.step(positions, now)

        left, right, state = self.black_line.step(positions, now)
        aligned = max(abs(value) for value in values) <= GAP_ALIGNMENT
        straight = len(values) >= 2 and max(values) - min(values) <= GAP_STRAIGHT_SPREAD
        if state == "FOLLOW" and near is not None and aligned and straight:
            if self.straight_seen is None or now - self.straight_seen > GAP_APPROACH_MAX_AGE:
                self.straight_frames = 0
            self.straight_frames += 1
            self.straight_seen = now
            correction = clamp((left - right) / 2.0, -GAP_STEER_LIMIT, GAP_STEER_LIMIT)
            self.saved_correction = 0.5 * self.saved_correction + 0.5 * correction
        elif state != "FOLLOW" or not aligned or (len(values) >= 2 and not straight):
            self.straight_frames = 0
            self.straight_seen = None
        # A short single-band transition can occur as the line exits the image.
        # Preserve earlier evidence only until GAP_APPROACH_MAX_AGE expires.
        return left, right, state


## 5. GreenMarkerTurnController: respond to green markers
Owns marker confirmation, stop/advance/turn sequence and cooldown. Returns None when it is not controlling motion.


In [ ]:
# Initial trial values, not calibrated turn angles or distances.
GREEN_CONFIRM_FRAMES = 3
GREEN_STOP_TIME = 0.12
GREEN_ADVANCE_TIME = 0.15 # Advance toward junction after stopping; tune from preview.
GREEN_ADVANCE_SPEED = BASE_SPEED
GREEN_TURN_SPEED = 0.15
GREEN_TURN_MIN_TIME = 0.25
GREEN_TURN_TIMEOUT = 1.5
UTURN_MIN_TIME = 0.8
UTURN_TIMEOUT = 2.5
GREEN_REACQUIRE_FRAMES = 3
GREEN_CLEAR_FRAMES = 5
GREEN_COOLDOWN = 0.8

def line_centered(positions):
    return all(positions.get(name) is not None and abs(positions[name]) < TURN_EXIT
               for name in ('near', 'middle'))

class GreenMarkerTurnController:
    """Confirm green instructions and execute them; return None when not in control."""
    def __init__(self):
        self.state = 'FOLLOW'
        self.state_started = None
        self.direction = None
        self.candidate = None
        self.candidate_frames = 0
        self.armed = True
        self.clear_frames = 0
        self.cooldown_until = 0.0
        self.departed = False
        self.reacquire_frames = 0
        self.stopped = False
        self.reason = ''

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        return 0.0, 0.0, 'STOP: ' + reason

    def enter(self, state, now):
        self.state = state
        self.state_started = now

    def turn(self):
        speed = clamp(GREEN_TURN_SPEED, 0.0, MAX_SPEED)
        if self.direction == 'RIGHT':
            return speed, -speed, 'GREEN_TURN_RIGHT'
        return -speed, speed, 'GREEN_U_TURN' if self.direction == 'BOTH' else 'GREEN_TURN_LEFT'

    def step(self, positions, direction, now, can_start=True):
        if self.stopped:
            return self.stop(self.reason)

        if self.state == 'GREEN_STOP':
            if now - self.state_started < GREEN_STOP_TIME:
                return 0.0, 0.0, 'GREEN_STOP'
            self.enter('GREEN_ADVANCE', now)

        if self.state == 'GREEN_ADVANCE':
            if now - self.state_started < GREEN_ADVANCE_TIME:
                speed = clamp(GREEN_ADVANCE_SPEED, 0.0, MAX_SPEED)
                return speed, speed, 'GREEN_ADVANCE'
            self.departed = False
            self.reacquire_frames = 0
            self.enter('GREEN_TURN', now)

        if self.state == 'GREEN_TURN':
            elapsed = now - self.state_started
            limit = UTURN_TIMEOUT if self.direction == 'BOTH' else GREEN_TURN_TIMEOUT
            minimum = UTURN_MIN_TIME if self.direction == 'BOTH' else GREEN_TURN_MIN_TIME
            if elapsed >= limit:
                return self.stop('green turn timed out')
            centered = line_centered(positions)
            if not centered:
                self.departed = True
            if self.departed and elapsed >= minimum and centered:
                self.reacquire_frames += 1
            else:
                self.reacquire_frames = 0
            if self.reacquire_frames < GREEN_REACQUIRE_FRAMES:
                return self.turn()
            # The coordinator resets line/gap history after GREEN_DONE.
            self.enter('FOLLOW', now)
            self.cooldown_until = now + GREEN_COOLDOWN
            self.clear_frames = 0
            return 0.0, 0.0, 'GREEN_DONE'

        if not self.armed:
            # Require actual absence, not UNKNOWN/CENTER, to clear the old marker.
            self.clear_frames = self.clear_frames + 1 if direction == 'NONE' else 0
            if now >= self.cooldown_until and self.clear_frames >= GREEN_CLEAR_FRAMES:
                self.armed = True

        # Do not initiate another manoeuvre in the middle of a gap or sharp turn.
        eligible = (self.armed and can_start and direction in ('LEFT','RIGHT','BOTH'))
        if eligible:
            if direction == self.candidate:
                self.candidate_frames += 1
            else:
                self.candidate, self.candidate_frames = direction, 1
            if self.candidate_frames >= GREEN_CONFIRM_FRAMES:
                self.direction = direction
                self.armed = False
                self.clear_frames = 0
                self.candidate = None
                self.candidate_frames = 0
                self.enter('GREEN_STOP', now)
                return 0.0, 0.0, 'GREEN_STOP'
        else:
            self.candidate = None
            self.candidate_frames = 0

        return None  # No green maneuver: coordinator chooses gap/black-line control.


## 6. NavigationCoordinator: the high-level decision point
Read this short class first to understand delegation. It gives green maneuvers priority, otherwise delegates to gap-aware black-line driving.


In [ ]:
class NavigationCoordinator:
    """Single decision point: green maneuver > gap crossing > black-line control.

    The runner handles camera validity and is the only writer to the motors.
    Controllers return commands; they do not directly access robot hardware.
    """
    def __init__(self):
        self.green_turn = GreenMarkerTurnController()
        self.gap = GapCrossingController()
        self.stopped = False
        self.reason = ''
        self.active_controller = 'BLACK_LINE'

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        self.active_controller = 'STOPPED'
        return 0.0, 0.0, 'STOP: ' + reason

    def step(self, positions, marker_direction, now):
        if self.stopped:
            return self.stop(self.reason)

        # A new green request must not interrupt an active gap or sharp bend.
        # An already active green maneuver continues regardless of this gate.
        can_start_green = (self.gap.gap_started is None and
                           self.gap.black_line.turn_started is None)
        command = self.green_turn.step(positions, marker_direction, now,
                                       can_start=can_start_green)
        if command is not None:
            self.active_controller = 'GREEN_TURN'
            if self.green_turn.stopped:
                # Preserve the baseline stop text and latch behavior.
                self.stopped = True
                self.reason = self.green_turn.reason
                self.active_controller = 'STOPPED'
            elif command[2] == 'GREEN_DONE':
                self.gap = GapCrossingController()
            return command

        # GapCrossingController checks gap evidence first, otherwise delegates
        # to its BlackLineController. This preserves v4's original priority.
        command = self.gap.step(positions, now)
        self.active_controller = ('GAP_CROSSING' if self.gap.gap_started is not None
                                  else 'BLACK_LINE')
        if self.gap.black_line.stopped:
            return self.stop(command[2])
        return command


## 7. Runner: observations → decision → motor commands
Checks camera freshness and duration, then displays the current owner and detail state. Preview never applies calculated drive commands.


In [ ]:
overlay_view = widgets.Image(format='jpeg', width=336, height=336)
mask_view = widgets.Image(format='jpeg', width=336, height=336)
status_view = widgets.Label(value='Ready for stationary preview')
display(widgets.VBox([widgets.HBox([overlay_view, mask_view]), status_view]))

def run_v9(seconds=10, drive=False):
    robot.stop()
    controller = NavigationCoordinator()
    deadline = time.monotonic() + seconds
    previous_stamp = None
    next_display = 0.0
    try:
        while time.monotonic() < deadline:
            sample = latest['sample']
            if sample is None:
                print('Stopped: no camera frame')
                break
            frame, captured_at = sample
            now = time.monotonic()
            # Check freshness BEFORE skipping repeated frames.
            if now - captured_at > FRAME_TIMEOUT:
                print('Stopped: stale camera frame')
                break
            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue
            previous_stamp = captured_at
            positions, overlay, _ = detect_curve(frame)
            markers, green_mask = detect_green(frame)
            height, width = frame.shape[:2]
            direction = classify_green_direction(positions, markers, width, height)
            now = time.monotonic()
            if now >= deadline or now - captured_at > FRAME_TIMEOUT:
                print('Stopped: deadline or slow image processing')
                break
            left, right, state = controller.step(positions, direction, now)
            if drive:
                robot.set_motors(left, right)
            if now >= next_display:
                points = get_line_points(positions, width, height)
                for a,b in zip(points, points[1:]):
                    cv2.line(overlay, tuple(map(int,a)), tuple(map(int,b)), (255,0,255), 2)
                for marker in markers:
                    x,y,w,h = marker['box']
                    cv2.rectangle(overlay, (x,y), (x+w,y+h), (0,255,0), 2)
                overlay_view.value = bgr8_to_jpeg(overlay)
                mask_view.value = bgr8_to_jpeg(cv2.cvtColor(green_mask, cv2.COLOR_GRAY2BGR))
                status_view.value = '%s | %s | marker=%s | L=%.2f R=%.2f' % (
                    'DRIVE' if drive else 'PREVIEW',state,direction,left,right) + ' | owner=' + controller.active_controller
                next_display = now + 0.1
            if controller.stopped:
                print(state)
                break
            time.sleep(0.01)
    finally:
        robot.stop()
        print('Run ended; motors stopped.')

def run_preview(seconds=20):
    run_v9(seconds=seconds, drive=False)

def run_robot(seconds=2):
    run_v9(seconds=seconds, drive=True)


## 8. Stationary preview
Confirm controller ownership and detailed status with the robot stationary or moved by hand.


In [ ]:
run_preview(seconds=20)


## 9. Short driving test
Begin with a single left marker, then right, then both. Start far enough before the marker for detection. Initial two-second tests may end before a whole turn completes; extend the overall duration gradually, not the failure timeout. Retest straight lines, curves, and gaps.


In [ ]:
# Explicitly uncomment to move the robot:
# run_robot(seconds=2)


## 10. Cleanup
Run after the driving/preview loop ends, before switching notebooks.


In [ ]:
robot.stop()
camera.unobserve(receive_frame, names="value")
camera_link.unlink()
camera.stop()
